In [ ]:
import run_design_pipeline
import parametrization_experiments.parametrization_experiment_helper as parametrization_experiment_helper
import time 
import parallelism, multiprocessing, itertools, setproctitle
import MeshFEM
import os

import inflation, sparse_matrices, mesh, numpy as np, pickle
import inflatables_parametrization as parametrization
from numpy.linalg import norm
from io_redirection import suppress_stdout
import visualization
import visualize_stiffness
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib as mpl
import utils, mesh_utilities, benchmark
import shapely
from mesh_utilities import SurfaceSampler, tubeRemesh
import py_newton_optimizer
# from tri_mesh_viewer import TriMeshViewer
from visualization import TriMeshViewerWithSurface
import boundaries
from matplotlib import pyplot as plt
import sheet_optimizer, opt_config
import fabrication
import py_newton_optimizer
import boundaries
import time
import os
import parallelism, multiprocessing, itertools, setproctitle
import serialization_helper
import sys

In [ ]:
# Update these!
shape_index = 1
pattern_index = 1

tag = 'fixed_boundary'

# The time stamp should be the one you used in `run_coarse_design.py`
output_time_stamp = 'demo_1'

# Find these by checking the files in `inverse_design/output/optimization/"output_time_stamp"/"shape_name_pattern_name"`
run_time_stamp = '2024_07_10_00_19'

In [ ]:

experiment_file, stiffness_path, pattern_name, num_pattern_params, param_index, default_param, param_range, param_normalization_factor, fusing_curve_polyline, shape_name, shape_path, use_holes = parametrization_experiment_helper.parse_input(shape_index, pattern_index)


In [ ]:
meshing_data_path = 'inverse_design/output/meshing_output_{}/{}_{}/'.format(output_time_stamp, shape_name, pattern_name)
optimization_data_path = 'inverse_design/output/optimization/{}/{}_{}/'.format(output_time_stamp, shape_name, pattern_name)

In [ ]:
# print("Load existing optimization for tag {}".format(tag))
# sheet_opt = sheet_optimizer.load(optimization_data_path + '2024_01_19_23_48_{}_{}_{}.pkl.gz'.format(tag, shape_name, pattern_name))
# targetAttractedSheet = sheet_opt.rso.targetAttractedInflation()

print("Load existing optimization for tag {}".format(tag))
sheet_opt = sheet_optimizer.load(optimization_data_path + '{}_{}_{}_{}.pkl.gz'.format(run_time_stamp, tag, shape_name, pattern_name))
targetAttractedSheet = sheet_opt.rso.targetAttractedInflation()

In [ ]:

tas = sheet_opt.rso.targetAttractedInflation()
tsf = tas.targetSurfaceFitter()
targetSurf = mesh.Mesh(tsf.targetSurfaceV, tsf.targetSurfaceF)


In [ ]:
from visualization import TriMeshViewerWithSurface

In [ ]:
isheet = sheet_opt.rso.sheet()
optMesh = sheet_opt.rso.mesh().copy()
origMesh = sheet_opt.rso.originalMesh().copy()


In [ ]:
viewer = TriMeshViewerWithSurface(isheet, targetSurf, width=768, height=640)
viewer.showWireframe(True)

# viewer.setCameraParams(((1.613494603240345, -3.9332708615926393, 1.4922998234349831),
# (-0.05948468564942635, 0.33267929672385665, 0.941162078339598),
# (0.0, 0.0, 0.0)))

viewer.update(scalarField=utils.getStrains(targetAttractedSheet.sheet())[:, 0])    

framerate = 10
def cb(it):
    if it % framerate == 0:
        viewer.update(scalarField=utils.getStrains(targetAttractedSheet.sheet())[:, 0])    



In [ ]:
viewer.show()

In [ ]:
viewer.update()

In [ ]:
fixedvars = boundaries.getOuterBoundaryVars(isheet)

In [ ]:
# Remove the target-attraction force and recompute the equilibrium
cr = inflation.inflation_newton(isheet, fixedVars = fixedvars, options = sheet_opt.opts, callback = cb)

In [ ]:
cr.success

## Optional Analysis

### Vibrational modes

In [ ]:
import compute_vibrational_modes

In [ ]:
lambdas, modes = compute_vibrational_modes.compute_vibrational_modes(isheet, mtype=compute_vibrational_modes.MassMatrixType.FULL, n=16, sigma=-1e-10, fixedVars = [])


In [ ]:
import mode_viewer, importlib
mview = mode_viewer.ModeViewer(isheet, modes, lambdas, amplitude=10000)
mview.show()

### Gravity

In [ ]:
isheet.rho = 1e-6
isheet.gravity = [0.0, -9.80635, 0.0]

In [ ]:
# sheet_var = np.load('output/meshing_output_low_res_2024_01_18_23_29/cashew_cosine_curve_amplitude_full_period/fixed_boundary_inflated_sheet_vars.npy')

In [ ]:
# targetAttractedSheet.sheet().setVars(sheet_var)

In [ ]:
# # Remove the target-attraction force and recompute the equilibrium
# targetAttractedSheet.fittingWeight = 1e-8
# inflation.inflation_newton(targetAttractedSheet, sheet_opt.rso.fixedEquilibriumVars(), sheet_opt.opts, callback = cb)


In [ ]:
isheet = sheet_opt.rso.sheet()
optMesh = sheet_opt.rso.mesh().copy()
origMesh = sheet_opt.rso.originalMesh().copy()


In [ ]:
# fusing_data = np.load('output/meshing_output_low_res_2024_01_18_23_29/cashew_cosine_curve_amplitude_full_period/fusing_data.npy')
fusing_data = np.load('output/meshing_output_low_res_2024_01_18_23_29/igloo_cosine_curve_amplitude_full_period//fusing_data.npy')

In [ ]:
run_time_stamp = time.strftime("%Y_%m_%d_%H_%M")

In [ ]:
import parametrization_helper

In [ ]:
importlib.reload(parametrization_helper)

In [ ]:

channelMargin = 0
final_vertices, concatenated_polylines = parametrization_helper.get_fabrication_file_from_mesh(sheet_opt.rso.sheet(), sheet_opt.rso.sheet().mesh(), fusing_data, channelMargin, [1e-4, 1e-4, 1e-4], optimization_data_path + '{}_{}_{}_{}_design_optimized_sheet_pattern_margin_{}.obj'.format(run_time_stamp, tag, shape_name, pattern_name, channelMargin))
visualization.plot_line_segments(final_vertices, concatenated_polylines, width = 20, height = 20, path = optimization_data_path + '{}_{}_{}_{}_design_optimized_sheet_pattern_margin_{}.png'.format(run_time_stamp, tag, shape_name, pattern_name, channelMargin))


In [ ]:

channelMargin = 0
final_vertices, concatenated_polylines = parametrization_helper.get_fabrication_file_from_mesh(sheet_opt.rso.sheet(), sheet_opt.rso.sheet().mesh(), fusing_data, channelMargin, [1e-4, 1e-4, 1e-4], optimization_data_path + '{}_{}_{}_{}_design_optimized_sheet_pattern_margin_{}.svg'.format(run_time_stamp, tag, shape_name, pattern_name, channelMargin), use_obj = False)

In [ ]:
importlib.reload(parametrization_helper)

In [ ]:
parametrization_helper.export_top_bottom_mesh(isheet, optimization_data_path, 'igloo', 'cosine_curve')

In [ ]:
targetSurf.save(optimization_data_path + '/igloo_cosine_curve_target_surface.obj')